In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack

In [2]:
df = pd.read_csv('../phishing_dataset.csv')
df.head

<bound method NDFrame.head of             sender_name                sender_email  \
0          Allison Hill    donaldgarcia@hotmail.com   
1     Caitlin Henderson            eric51@james.com   
2        Thomas Bradley         jason76@hotmail.com   
3        Brittany Moore          ycarlson@yahoo.com   
4         Pamela Romero  harrellkenneth@hotmail.com   
...                 ...                         ...   
9995       Tammy Oliver           james72@yahoo.com   
9996      Meagan Stuart        davisjames@yahoo.com   
9997    Jeanette Holmes    sandovaldouglas@wood.com   
9998     Patrick Taylor      tsmith@hill-dorsey.biz   
9999     Jeremy Griffin          nroberts@gross.org   

                                                subject  \
0        URGENT: Quality-focused background parallelism   
1           Enough analysis least by two bad fall pick.   
2         Attention: Persevering client-server firmware   
3             DocuSign: Right-sized maximized moderator   
4     Billing 

# **Using Logical Regression**

In [3]:
y = df['is_phishing'].astype(int)
y

0       1
1       0
2       1
3       1
4       1
       ..
9995    1
9996    1
9997    0
9998    0
9999    0
Name: is_phishing, Length: 10000, dtype: int64

In [4]:
df["Combined_text"] = df["sender_email"]+' '+df["subject"]+' '+df['body']
vectorizer = TfidfVectorizer(stop_words='english',max_features=5000)
text = vectorizer.fit_transform(df['Combined_text'])


if_url = df["contains_url"].astype(int)
url = if_url.values.reshape(-1,1)

x = hstack([text,url])

#x = text

In [5]:
x_train,x_temp,y_train,y_temp = train_test_split(x,y,test_size=0.3,random_state=42)
x_verif,x_test,y_verif,y_test = train_test_split(x_temp,y_temp,test_size=0.5)

In [6]:
best_verif_score = 0
best_model = None
best_setting = 0

settings_to_test = [0.01, 0.1, 1.0, 10.0, 100.0]

for curr_c in settings_to_test:
    model = LogisticRegression(C=curr_c,max_iter=1000)
    model.fit(x_train,y_train)
    
    y_verif_pred = model.predict(x_verif)
    curr_score = accuracy_score(y_verif,y_verif_pred)
    
    print("*"*30)
    print(f"C Setting: {curr_c} verificatuon score: {curr_score*100:.2f}")
    
    
    if curr_score > best_verif_score:
        best_verif_score = curr_score
        best_model = model
        best_setting = curr_c
        
print(f"\n\nBest C setting: {best_setting} best verification score: {best_verif_score}")

******************************
C Setting: 0.01 verificatuon score: 78.00
******************************
C Setting: 0.1 verificatuon score: 100.00
******************************
C Setting: 1.0 verificatuon score: 100.00
******************************
C Setting: 10.0 verificatuon score: 100.00
******************************
C Setting: 100.0 verificatuon score: 100.00


Best C setting: 0.1 best verification score: 1.0


In [7]:
# 1. Check your dataset balance
print("Dataset Breakdown:")
print(df['is_phishing'].value_counts(normalize=True) * 100)

# 2. Look at a detailed report for one of your models
from sklearn.metrics import classification_report
print("\nDetailed Report:")
print(classification_report(y_verif, best_model.predict(x_verif)))

Dataset Breakdown:
is_phishing
True     50.33
False    49.67
Name: proportion, dtype: float64

Detailed Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       750
           1       1.00      1.00      1.00       750

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       1.00      1.00      1.00      1500



In [8]:
y_test_pred = model.predict(x_test)
print("=== FINAL TEST RESULTS===")
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_test_pred))

=== FINAL TEST RESULTS===
Test Accuracy: 100.00%

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       763
           1       1.00      1.00      1.00       737

    accuracy                           1.00      1500
   macro avg       1.00      1.00      1.00      1500
weighted avg       1.00      1.00      1.00      1500

